In [1]:
import pandas as pd 
import numpy as np 
from  sklearn.model_selection import train_test_split

df = pd.read_csv('../data/tracks_clean.csv')
df_unique = df.drop_duplicates(subset='track_id', keep='first').copy()

print('Unique tracks:', df_unique.shape[0])

Unique tracks: 89023


## Setup — Load and Deduplicate

Loads the Week 1 cleaned dataset and deduplicates by track_id, same
leakage guard established in Week 3 (a song counted once per genre tag
would otherwise appear in both train and test).

**Result:** 89,023 unique tracks — matches every prior notebook exactly.

In [2]:
df_unique['contrast'] = df_unique['energy'] * (1 - df_unique['acousticness'])
df_unique['tension'] = (1 - df_unique['valence']) * df_unique['energy']
df_unique['impact'] = df_unique['energy'] * df_unique['danceability'] * (df_unique['loudness']  + 60) / 60

## Rebuild Engineered Features

Recreates contrast, tension, and impact with the same formulas from
Week 3, now as model inputs rather than correlation tests.

In [3]:
df_unique['hit'] = (df_unique['popularity'] >= 70).astype(int)

## Define the Target Variable

hit = 1 if popularity >= 70, else 0 — same threshold established and
sensitivity-tested in Week 2 (Q11, Q14).

In [4]:
print(df_unique['hit'].value_counts())
print('\nPercentage')
print(df_unique['hit'].value_counts(normalize=True) * 100)

hit
0    85899
1     3124
Name: count, dtype: int64

Percentage
hit
0    96.490795
1     3.509205
Name: proportion, dtype: float64


## Q20 — What is the class balance?

**Result:** 85,899 non-hits (96.49%) vs. 3,124 hits (3.51%). Matches
peak_70 from Week 3 exactly (3,124 of 89,023).

**Decision:** this imbalance shapes every choice from here forward —
stratified splitting, class weighting, and threshold tuning are all
direct responses to this number.

In [5]:
from sklearn.model_selection import train_test_split

feature_cols = ['danceability', 'energy', 'loudness', 'speechiness', 
                 'acousticness', 'instrumentalness', 'liveness', 
                 'valence', 'tempo', 'duration_ms', 'key', 'mode', 
                 'explicit', 'contrast', 'tension', 'impact']

X = df_unique[feature_cols]
y = df_unique['hit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train hits:", y_train.mean().round(4) * 100, "%")
print("Test hits:", y_test.mean().round(4) * 100, "%")

Train hits: 3.51 %
Test hits: 3.51 %


## Train/Test Split

16 audio + engineered features (feature_cols). Split 80/20, stratified
by hit so both sides preserve the 3.51% ratio.

**Result:** Train hits 3.51%, Test hits 3.51% — confirmed.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Train the baseline model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1 Score:", round(f1_score(y_test, y_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9631
Precision: 0.2647
Recall: 0.0288
F1 Score: 0.0519

Confusion Matrix:
[[17130    50]
 [  607    18]]


## Q21 — Does a baseline Random Forest predict hits? (H1, first pass)

Default RandomForestClassifier, default 0.5 decision threshold.

**Result:** Accuracy 96.31%, Precision 26.47%, Recall 2.88%, F1 0.052.
Confusion matrix: only 18 of 625 real hits caught.

**Finding:** accuracy is misleading here — a model that always predicts
"non-hit" would score ~96.5% without learning anything. Recall is the
metric that matters, and it's near zero.

In [7]:
model_balanced = RandomForestClassifier(
    n_estimators=100, 
    random_state=42,
    class_weight='balanced'
)
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred_balanced), 4))
print("Precision:", round(precision_score(y_test, y_pred_balanced), 4))
print("Recall:", round(recall_score(y_test, y_pred_balanced), 4))
print("F1 Score:", round(f1_score(y_test, y_pred_balanced), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

Accuracy: 0.959
Precision: 0.1329
Recall: 0.0304
F1 Score: 0.0495

Confusion Matrix:
[[17056   124]
 [  606    19]]


## class_weight='balanced' — does it help?

**Result:** Recall improves marginally (2.88% → 3.04%, 18→19 hits
caught); Precision drops (26.5% → 13.3%).

**Finding:** class_weight adjusts how training penalizes errors on the
minority class, but can't manufacture signal that isn't present in the
features. The improvement is negligible.

In [8]:
# Probabilidades predichas (no solo la clasificación final 0/1)
y_proba = model_balanced.predict_proba(X_test)[:, 1]

# Comparar la distribución de probabilidad para hits reales vs no-hits reales
import pandas as pd
proba_df = pd.DataFrame({'actual': y_test, 'predicted_proba': y_proba})

print("Probabilidad promedio asignada a HITS reales:", 
      round(proba_df[proba_df['actual']==1]['predicted_proba'].mean(), 4))
print("Probabilidad promedio asignada a NO-HITS reales:", 
      round(proba_df[proba_df['actual']==0]['predicted_proba'].mean(), 4))

Probabilidad promedio asignada a HITS reales: 0.0832
Probabilidad promedio asignada a NO-HITS reales: 0.0352


## Diagnostic — Raw Probabilities vs. the 0.5 Cutoff

predict_proba() returns the model's raw score before the 0.5 cutoff is
applied.

**Result:** real hits average 0.083 probability; real non-hits average
0.035 — a 2.4x separation, but both far below 0.5.

**Finding:** the model IS distinguishing hits from non-hits — the
problem is that with a 3.5% base rate, even correct signal rarely
crosses 0.5. This motivates threshold tuning instead of the default
cutoff.

In [9]:
from sklearn.metrics import precision_recall_curve

# Probabilidades ya calculadas del modelo balanceado
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Probar algunos umbrales concretos, más cercanos a donde vive la señal
for t in [0.5, 0.3, 0.15, 0.1, 0.08, 0.05]:
    y_pred_t = (y_proba >= t).astype(int)
    prec = precision_score(y_test, y_pred_t, zero_division=0)
    rec = recall_score(y_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_test, y_pred_t, zero_division=0)
    print(f"Threshold {t}: Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}")

Threshold 0.5: Precision=0.130, Recall=0.030, F1=0.049
Threshold 0.3: Precision=0.106, Recall=0.035, F1=0.053
Threshold 0.15: Precision=0.126, Recall=0.112, F1=0.119
Threshold 0.1: Precision=0.112, Recall=0.240, F1=0.153
Threshold 0.08: Precision=0.095, Recall=0.315, F1=0.146
Threshold 0.05: Precision=0.077, Recall=0.523, F1=0.134


## Threshold Tuning

Tested thresholds from 0.5 down to 0.05 using precision_recall_curve
and manual sweeps.

**Result:** best F1 at threshold 0.1 (Precision 0.112, Recall 0.240,
F1 0.153) — 3x better than the 0.5 default (F1 0.049).

**Decision:** 0.1 adopted as the working threshold for this model going
forward, chosen for F1 balance. A lower threshold (0.05) trades more
precision for much higher recall (52.3%) and may suit the business case
better, since a human editorial team reviews candidates afterward.

In [10]:


# Feature importances del modelo balanceado
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_balanced.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

             feature  importance
5   instrumentalness    0.094870
9        duration_ms    0.081686
2           loudness    0.081447
4       acousticness    0.077683
6           liveness    0.076705
14           tension    0.073701
1             energy    0.070199
13          contrast    0.068985
3        speechiness    0.068773
8              tempo    0.066388
0       danceability    0.064610
7            valence    0.062029
15            impact    0.061421
10               key    0.032464
12          explicit    0.012193
11              mode    0.006847


## Q22 — Which features does the model actually use?

**Result:** instrumentalness ranks #1 (0.095) — well ahead of
danceability (#11, 0.065), which led every Week 3 correlation test.

**Finding:** correlation (linear, single-variable) and feature
importance (non-linear, tree-based) measure different things. Worth
investigating why instrumentalness dominates before trusting the
ranking.

In [11]:
print("Instrumentalness promedio en HITS:", 
      round(df_unique[df_unique['hit']==1]['instrumentalness'].mean(), 4))
print("Instrumentalness promedio en NO-HITS:", 
      round(df_unique[df_unique['hit']==0]['instrumentalness'].mean(), 4))

Instrumentalness promedio en HITS: 0.0337
Instrumentalness promedio en NO-HITS: 0.1758


## Investigating instrumentalness — Hit vs. Non-Hit

**Result:** hits average 0.034 instrumentalness; non-hits average
0.176 — a 5x gap, sharper than the -0.12 linear correlation from
Week 3 suggested.

In [12]:

top_instrumental_genres = df_unique.groupby('track_genre')['instrumentalness'].mean().sort_values(ascending=False)
print("Top 10 géneros con MÁS instrumentalness:")
print(top_instrumental_genres.head(10))
print("\nTop 10 géneros con MENOS instrumentalness:")
print(top_instrumental_genres.tail(10))

Top 10 géneros con MÁS instrumentalness:
track_genre
study             0.789496
minimal-techno    0.747205
sleep             0.733762
new-age           0.714363
detroit-techno    0.691407
idm               0.675404
ambient           0.675273
classical         0.604044
iranian           0.591631
techno            0.558232
Name: instrumentalness, dtype: float64

Top 10 géneros con MENOS instrumentalness:
track_genre
gospel       0.004029
latin        0.003778
country      0.002479
forro        0.001253
reggae       0.001155
party        0.001125
reggaeton    0.000984
comedy       0.000364
sertanejo    0.000192
pagode       0.000069
Name: instrumentalness, dtype: float64


## Investigating instrumentalness — By Genre

**Result:** highest-instrumentalness genres are non-vocal/functional
(study, minimal-techno, sleep, ambient, classical). Lowest are
vocal-driven (reggaeton, sertanejo, gospel, latin) — the same genres
that dominated Week 2's hit rankings.

**Finding:** instrumentalness is not an independent audio signal — it's
a proxy for genre. The model found genre information through a side
door before genre was ever given directly. Motivates the direct test
in H2 below.

In [13]:
# One-hot encode track_genre
genre_dummies = pd.get_dummies(df_unique['track_genre'], prefix='genre')
print("Genre columns created:", genre_dummies.shape[1])

# Combine with existing features
X_with_genre = pd.concat([df_unique[feature_cols], genre_dummies], axis=1)
y = df_unique['hit']

print("Total features now:", X_with_genre.shape[1])

Genre columns created: 113
Total features now: 129


## One-Hot Encode track_genre (H2 setup)

pd.get_dummies converts the 114 genre labels into 113 binary columns
(one dropped by default — an all-zero row across the 113 already
identifies the missing category).

**Result:** 129 total features (16 audio + 113 genre).

In [14]:
X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_with_genre, y, test_size=0.2, random_state=42, stratify=y
)

model_with_genre = RandomForestClassifier(
    n_estimators=100, random_state=42, class_weight='balanced'
)
model_with_genre.fit(X_train_g, y_train_g)

y_pred_g = model_with_genre.predict(X_test_g)
y_proba_g = model_with_genre.predict_proba(X_test_g)[:, 1]

## Train with Genre Included

Same split logic as Cell 4 (random_state=42, stratify=y) on the
129-column feature set, for a fair comparison against the no-genre
model.

In [15]:
threshold = 0.1
y_pred_g_t = (y_proba_g >= threshold).astype(int)

print("=== CON GÉNERO (threshold 0.1) ===")
print("Precision:", round(precision_score(y_test_g, y_pred_g_t), 4))
print("Recall:", round(recall_score(y_test_g, y_pred_g_t), 4))
print("F1:", round(f1_score(y_test_g, y_pred_g_t), 4))

=== CON GÉNERO (threshold 0.1) ===
Precision: 0.2454
Recall: 0.4912
F1: 0.3273


## Q23 — Does genre improve the model? (H2, direct test)

Same threshold (0.1) as the no-genre model, for a clean comparison.

**Result:** Precision 0.245, Recall 0.491, F1 0.327 — every metric
roughly doubles vs. the no-genre model (F1 0.153 → 0.327).

**Finding (H2):** NOT supported as originally framed ("genre is
secondary"). Genre carries substantial, non-redundant predictive
signal.

In [16]:
importances_g = pd.DataFrame({
    'feature': X_with_genre.columns,
    'importance': model_with_genre.feature_importances_
}).sort_values('importance', ascending=False)

print(importances_g.head(20))


             feature  importance
5   instrumentalness    0.068331
9        duration_ms    0.057705
2           loudness    0.056285
4       acousticness    0.056185
14           tension    0.055437
13          contrast    0.054848
6           liveness    0.052434
1             energy    0.050991
15            impact    0.048389
0       danceability    0.047066
8              tempo    0.046377
3        speechiness    0.045320
7            valence    0.044189
10               key    0.026298
36       genre_dance    0.018754
81       genre_k-pop    0.018280
18    genre_alt-rock    0.012842
84      genre_latino    0.012229
12          explicit    0.011875
73   genre_indie-pop    0.010761


## Feature Importance, With Genre

**Result:** instrumentalness still ranks #1 (0.068). The strongest
individual genre, genre_dance, ranks #15 (0.019) — far behind every
audio feature in the top 14.

**Finding:** no single genre dominates. Whatever advantage genre
provides comes from combined weight across many genre columns, not one
powerful category.

In [17]:
genre_cols = [c for c in importances_g['feature'] if c.startswith('genre_')]
audio_cols = [c for c in importances_g['feature'] if not c.startswith('genre_')]

genre_total = importances_g[importances_g['feature'].isin(genre_cols)]['importance'].sum()
audio_total = importances_g[importances_g['feature'].isin(audio_cols)]['importance'].sum()

print(f"Suma importancia GÉNERO (113 cols): {genre_total:.3f}")
print(f"Suma importancia AUDIO (16 cols): {audio_total:.3f}")

Suma importancia GÉNERO (113 cols): 0.271
Suma importancia AUDIO (16 cols): 0.729


## Audio vs. Genre — Total Importance

**Result:** the 16 audio features sum to 72.9% of total importance;
the 113 genre columns sum to 27.1%.

**Finding (H2, refined):** audio still carries the majority of the
model's decision weight — nearly 3x more than genre. Genre doesn't need
to be the majority to unlock large performance gains (Cell 14): it
appears to add information genuinely absent from the audio features
tested, rather than out-competing them. The honest reading of H2 is
"not supported as secondary, but audio remains the foundation" — both
signals are real and complementary.

In [18]:
from xgboost import XGBClassifier

# Calcular el peso de balance manualmente (XGBoost usa scale_pos_weight, no class_weight)
scale_pos_weight = (y_train_g == 0).sum() / (y_train_g == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

model_xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)
model_xgb.fit(X_train_g, y_train_g)

y_proba_xgb = model_xgb.predict_proba(X_test_g)[:, 1]

scale_pos_weight: 27.5


## Second Algorithm — XGBoost (Robustness Check)

scale_pos_weight computed directly from the training data (27.5) —
XGBoost's equivalent of class_weight='balanced'. Trained on the same
129-column, genre-included data as Cell 13, for a fair comparison.

In [19]:
threshold = 0.1
y_pred_xgb_t = (y_proba_xgb >= threshold).astype(int)

print("=== XGBoost CON GÉNERO (threshold 0.1) ===")
print("Precision:", round(precision_score(y_test_g, y_pred_xgb_t), 4))
print("Recall:", round(recall_score(y_test_g, y_pred_xgb_t), 4))
print("F1:", round(f1_score(y_test_g, y_pred_xgb_t), 4))

=== XGBoost CON GÉNERO (threshold 0.1) ===
Precision: 0.0742
Recall: 0.9744
F1: 0.138


## XGBoost at Threshold 0.1 — Why That's the Wrong Comparison

**Result:** Precision 0.074, Recall 0.974, F1 0.138 — the model is
calling almost every song a hit.

**Finding:** reusing Random Forest's optimal threshold (0.1) on XGBoost
was invalid. scale_pos_weight=27.5 pushes XGBoost's probabilities to a
different, generally higher scale than Random Forest's — the two
models' scores aren't directly comparable at a shared fixed threshold.

In [20]:
from sklearn.metrics import roc_auc_score

auc_rf = roc_auc_score(y_test_g, y_proba_g)
auc_xgb = roc_auc_score(y_test_g, y_proba_xgb)

print("AUC Random Forest (con género):", round(auc_rf, 4))
print("AUC XGBoost (con género):", round(auc_xgb, 4))

AUC Random Forest (con género): 0.8458
AUC XGBoost (con género): 0.8807


## Comparing Algorithms Fairly — AUC-ROC

AUC-ROC measures ranking quality across all thresholds at once,
sidestepping the threshold-mismatch problem from Cell 18.

**Result:** Random Forest AUC 0.846; XGBoost AUC 0.881.

**Finding:** both comfortably above 0.5 (random guessing) — real,
usable signal exists. XGBoost ranks modestly better overall.

In [21]:
for t in [0.5, 0.7, 0.8, 0.85, 0.9, 0.95]:
    y_pred_t = (y_proba_xgb >= t).astype(int)
    prec = precision_score(y_test_g, y_pred_t, zero_division=0)
    rec = recall_score(y_test_g, y_pred_t, zero_division=0)
    f1 = f1_score(y_test_g, y_pred_t, zero_division=0)
    print(f"Threshold {t}: Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}")
    

Threshold 0.5: Precision=0.126, Recall=0.730, F1=0.214
Threshold 0.7: Precision=0.203, Recall=0.498, F1=0.288
Threshold 0.8: Precision=0.260, Recall=0.347, F1=0.297
Threshold 0.85: Precision=0.330, Recall=0.277, F1=0.301
Threshold 0.9: Precision=0.403, Recall=0.157, F1=0.226
Threshold 0.95: Precision=0.455, Recall=0.016, F1=0.031


## XGBoost's Own Optimal Threshold

Swept thresholds 0.5–0.95 (XGBoost's probabilities run higher than
Random Forest's, per Cell 18's diagnosis).

**Result:** best F1 at threshold 0.85 (Precision 0.330, Recall 0.277,
F1 0.301) — close to, but not better than, Random Forest's F1 0.327 at
its own optimal threshold.

**Finding:** two different algorithms converge on a similar performance
ceiling (F1 ~0.30–0.33, AUC ~0.85–0.88). This is evidence the ceiling
reflects the available signal in the data, not a weakness specific to
either algorithm.

**Decision:** Random Forest kept as the primary model for its stronger
recall, which better fits the Business Problem — surfacing overlooked
tracks matters more than avoiding false positives, since a human team
reviews candidates afterward.

In [22]:


dashboard_cols = ['track_name', 'artists', 'track_genre', 'popularity', 'hit',
                   'energy', 'danceability', 'valence', 'instrumentalness',
                   'loudness', 'duration_ms', 'explicit', 
                   'contrast', 'tension', 'impact']

df_unique[dashboard_cols].to_csv('tableau_tracks.csv', index=False)
print("Guardado:", df_unique[dashboard_cols].shape)

Guardado: (89023, 15)


In [23]:
# 2. 
importances_g.to_csv('tableau_feature_importance.csv', index=False)

model_summary = pd.DataFrame({
    'model': ['Random Forest (audio only)', 'Random Forest (+ genre)', 'XGBoost (+ genre)'],
    'auc': [None, 0.846, 0.881],
    'f1': [0.153, 0.327, 0.301]
})
model_summary.to_csv('tableau_model_results.csv', index=False)

## Week 4 Summary — Modeling

**H1 (pattern exists):** partial support. Random Forest and XGBoost
both separate hits from non-hits meaningfully (AUC 0.85–0.88), but
absolute performance is modest (best F1 ~0.30–0.33) — consistent with
Week 3's finding of diffuse, non-dominant signal.

**H2 (genre is secondary):** not supported. Adding track_genre roughly
doubled every metric (F1 0.153 → 0.327). Audio features still carry
more total model weight (73% vs. 27%), but genre adds signal audio
alone didn't have.

**H3 (cohesion matters):** partial support. contrast and tension
outranked danceability in feature importance (unlike Week 3's linear
correlations), suggesting the engineered features do capture real,
non-linear interactions a Random Forest can exploit — even though they
underperformed in simple correlation.

**Model selection:** Random Forest with track_genre included, threshold
0.1, kept as the primary model for its recall advantage over XGBoost at
comparable F1.